## DoubleML

This notebook is a hands-on introduction to Double-ML using synthetic data. The aim is for me to get a greater understanding of DoubleML and to motivate why DoubleML is needed.


*Results*  
A naive regression of Y on D estimates a coefficient of about 4.06, which is biased upwards (from the true value of 3.0). However this example probably isn't the best, DML works and agrees with OLS when the linear model is correctly specified.

In [1]:
import numpy as np
import statsmodels.api as sm # for OLS regression

from sklearn.ensemble import RandomForestRegressor # Y is continuous so we use a regressor here
from sklearn.ensemble import RandomForestClassifier # D is binary so we use a classifier here

from sklearn import model_selection # needed for k-folds

In [2]:
# build the data generating process (DGP). Before estimating
# anything, we need synthetic data where we know the ground truth.

# simulate the causal structure
# X -> D
# X -> Y
# D -> Y

In order to prove that OLS doesn't work we need either:  
A) Nonlinear confounding that interacts with treatment assignment in a way linear regression cannot partial out   
B) Treatment effect heterogeneity correlated with treatment assignment.  

A more in-depth description about this can be found in the report.

#### Data Generating Process

In [3]:
def generate_data(n=2000,
                  n_features=20,
                  true_effect=3.0,
                  seed=42):
  """
  Description
  -----------
  The first 3 cols of X appear in both the treatment model
  and the outcome model. That's exactly what creates the bias
  a naive regression would suffer from.

  Features 4-10 are pure noise.

  Inputs
  ------
  n: number of obvs
  n_features: num of confounding features in X
  true_effect: the TRUE causal effect of D on Y (theta*)
  seed: random seed for reproducibility

  Returns
  -------
  Y: outcome (spending)
  D: treatment (ad shown: 0 or 1)
  X: confounders (user features)
  """

  # rng is a random number generator object
  rng = np.random.default_rng(seed)

  # confounders - user features
  # draws random numbers from the standard normal distribution
  # shapes them into an array of size (n, n_features)
  X = rng.standard_normal((n, n_features))

  # Treatment
  # P( D=1 | X) - targetting algorithm
  # Only the first 3 features actually drive targetting

  # D is some combination of the first 3 columns of X?


  signal = (
    np.sin(X[:,0])
    + X[:,1]**2
    + X[:,2]*X[:,3]
    + np.exp(0.5*X[:,4])
    + np.tanh(X[:,5])
  )

  # selection depends on interaction
  propensity = 1 / (1 + np.exp(-signal))
  D = rng.binomial(1, propensity)

  # building Y
  # Outcome model: E[Y | X] - baseline spending driven by features
  # only the first 3 features also drive spending
  baseline = (
    2*np.sin(X[:,0])
    + X[:,1]**2
    + X[:,2]*X[:,3]
    + np.exp(0.5*X[:,4])
    + np.tanh(X[:,5])
  )

  tau_x = 3 + X[:,0]

  Y = true_effect * D + baseline + rng.standard_normal(size=n)
  # true causal effect of the ad plus noise
  #noise = rng.standard_normal(n)
  #Y = tau_x * D + baseline_spending + noise

  return Y, D, X


# Monte Carlo Simulation

In [4]:
# number of runs
iter = 10            # for the real assessment this would need to be higher
true_effect = 3.0

# initialise variables
theta_hat_point_estimates = np.zeros((iter)) # DML
ols_point_estimates = np.zeros((iter))       # OLS

theta_hat_se = np.zeros(iter)
ols_se = np.zeros(iter)

theta_hat_lb = np.zeros(iter)
theta_hat_ub = np.zeros(iter)

ols_lb = np.zeros(iter)
ols_ub = np.zeros(iter)

dml_coverage = np.zeros(iter, dtype=bool)
ols_coverage = np.zeros(iter, dtype=bool)

true_theta_in_range = np.full(iter, False, dtype=bool)

for i in range(iter):

  print(f'Currently on iteration {i+1} of {iter}.')

  seed = i

  # generate the data
  Y, D, X = generate_data(n=4000,
                  n_features=100,
                  true_effect=3.0,
                  seed=seed)

  # fit the standard regression: Y ~ D + X

  X_ols = np.column_stack((D, X))
  X_ols = sm.add_constant(X_ols)

  ols_model = sm.OLS(Y, X_ols).fit(cov_type='HC3')

  # coefficient on D
  ols_theta = ols_model.params[1]
  ols_point_estimates[i] = ols_theta

  ols_std_error = ols_model.bse[1]
  ols_lower = ols_theta - 1.96 * ols_std_error
  ols_upper = ols_theta + 1.96 * ols_std_error

  ols_se[i] = ols_std_error
  ols_lb[i] = ols_lower
  ols_ub[i] = ols_upper

  if ols_lower <= true_effect <= ols_upper:
    ols_coverage[i] = True

  # run the full cross-fitting procedure
  # generate 5 reproducible splits

  kf = model_selection.KFold(n_splits=5,
                        shuffle=True,
                        random_state=seed)

  # initialise space for the residuals of each fold to be stored
  n = Y.shape[0]
  Y_res = np.zeros((n,))
  D_res = np.zeros((n,))

  propensity_score = np.zeros((n,))


  for train_index, test_index in kf.split(X):

    regressor = RandomForestRegressor(
        n_estimators=400,
        random_state=seed,
        n_jobs=-1,
        oob_score=False
    )

    classifier = RandomForestClassifier(
        n_estimators=400,
        random_state=seed,
        n_jobs=-1,
        oob_score=False
    )

    X_train = X[train_index]
    D_train = D[train_index]
    y_train = Y[train_index]

    X_test = X[test_index]
    D_test = D[test_index]
    y_test = Y[test_index]

    # fit a RF Regressor on the training X and training Y
    regressor.fit(X_train, y_train)
    y_pred = regressor.predict(X_test)

    Y_res[test_index] = y_test - y_pred

    classifier.fit(X_train, D_train)
    # predict class probabilities for D
    # we take the second col, where P(D=1|X)
    D_prob = classifier.predict_proba(X_test)[:,1]
    # computed as the mean predicted class probabilities of the trees
    # in the forest. The class prob of a single tree is the fraction
    # of samples of the same class in a leaf

    # adding to propensity score for heteroskedastic plot
    propensity_score[test_index] = D_prob

    D_res[test_index] = D_test - D_prob

  model = sm.OLS(Y_res, D_res).fit(cov_type='HC3')

  theta_hat = model.params[0]
  theta_hat_point_estimates[i] = theta_hat

  Psi = Y_res - theta_hat * D_res
  var_theta = 1/n * np.mean(D_res**2 * Psi**2) / np.mean(D_res**2) ** 2
  se = np.sqrt(var_theta)

  lb = theta_hat - 1.96 * se
  ub = theta_hat + 1.96 * se

  theta_hat_se[i] = se
  theta_hat_lb[i] = lb
  theta_hat_ub[i] = ub

  if lb <= true_effect <= ub:
      dml_coverage[i] = True

Currently on iteration 1 of 10.


Currently on iteration 2 of 10.
Currently on iteration 3 of 10.
Currently on iteration 4 of 10.
Currently on iteration 5 of 10.
Currently on iteration 6 of 10.
Currently on iteration 7 of 10.
Currently on iteration 8 of 10.
Currently on iteration 9 of 10.
Currently on iteration 10 of 10.


In [5]:
# ---------- DML ----------

dml_mean = np.mean(theta_hat_point_estimates)
dml_bias = dml_mean - true_effect
dml_sd = np.std(theta_hat_point_estimates, ddof=1)
dml_rmse = np.sqrt(
    np.mean((theta_hat_point_estimates - true_effect)**2)
)
dml_avg_se = np.mean(theta_hat_se)
dml_coverage_rate = np.mean(dml_coverage)

# ---------- OLS ----------

ols_mean = np.mean(ols_point_estimates)
ols_bias = ols_mean - true_effect
ols_sd = np.std(ols_point_estimates, ddof=1)
ols_rmse = np.sqrt(
    np.mean((ols_point_estimates - true_effect)**2)
)
ols_avg_se = np.mean(ols_se)
ols_coverage_rate = np.mean(ols_coverage)

In [6]:
import pandas as pd
results = pd.DataFrame({
    "Method": ["OLS", "DML"],
    "Mean Estimate": [ols_mean, dml_mean],
    "Bias": [ols_bias, dml_bias],
    "Monte Carlo SD": [ols_sd, dml_sd],
    "Average SE": [ols_avg_se, dml_avg_se],
    "RMSE": [ols_rmse, dml_rmse],
    "Coverage": [ols_coverage_rate, dml_coverage_rate]
})

print(results.round(3))

  Method  Mean Estimate   Bias  Monte Carlo SD  Average SE   RMSE  Coverage
0    OLS          4.624  1.624           0.063       0.073  1.625       0.0
1    DML          3.734  0.734           0.052       0.064  0.735       0.0
